In [ ]:
# === Setup (Google Colab) ===
# Run once. Uncomment the install line the first time you open the notebook.
# %pip install librosa soundfile -q

# All audio in this notebook is either synthesized in code or loaded from
# librosa's built-in example clips, so it runs anywhere -- no manual file paths.

# What is a Signal? 

In simple words, signal is anything that has some message. Be it physical stimuli, some eye-to-eye contact, some action, everything that gives a message can become a signal.

It's hard to broadly define what it is, because it has such a vast domain.
Signal can mean various kinds of signals - traffic signal, digital signal, analog signal, a beacon, a code - but it all boils down to the same fundamental thing. Signals convey messages. Can be between people, between computers, between machines and humans, between animals, animals and humans - anything.

## Technical definition of Signal

A Signal is any "entity" that changes over a certain measurable "parameter". That entity is usually represented as a function of the parameter which can be time, space or anything that is quantifiable.

For example, we sin(wt) is a function of t which can represented as a signal. An image is also a signal, which is represented in a 2D grid of numeric (pixel) values as a function of space.


**Is a Audio a Signal**?
Yes, Audio is a Signal. It is a continuous variation of sound, this variation when measured creates waves also called sound waves.

# What is an Audio Signal? 
Audio signals are sound signals, defined as pressure variations travelling through the air. These variations in pressure can be described as waves and correspondingly they are often called sound waves.

In [ ]:
# Import necessary libraries
import numpy as np # Numerical computing and array operations
import librosa # Audio and music processing
import librosa.display # Data visualization
import matplotlib.pyplot as plt
import IPython.display as ipd # Interactive display of audio in Jupyter notebooks
from scipy.io import wavfile # Audio file I/O operations
import pandas as pd # Data manipulation and analysis
from scipy.signal import lfilter  # for the synthesized-sound demos


In [ ]:
def play_and_show(y, sr, title, fmax=None):
    """Play an audio clip and show its spectrogram underneath."""
    ipd.display(ipd.Audio(y, rate=sr))
    plt.figure(figsize=(11, 4))
    D = librosa.amplitude_to_db(np.abs(librosa.stft(y)), ref=np.max)
    librosa.display.specshow(D, sr=sr, x_axis="time", y_axis="hz", cmap="magma")
    if fmax:
        plt.ylim(0, fmax)
    plt.colorbar(format="%+2.0f dB")
    plt.title(title)
    plt.tight_layout()
    plt.show()

In [ ]:
# Load a musical example clip from librosa (downloads once, then cached).
# We take the first 20 seconds to keep the plots readable.
audio_path = librosa.example('nutcracker')   # Tchaikovsky - orchestral, tonal
y, sr = librosa.load(audio_path, duration=20)

In [ ]:
#print(type(y))

print(f"The sample: {y}")
print(f"The dimension of the samples: {y.ndim}")
#print(type(sr))

librosa.load() reads the audio path and returns two variable:
* y - Number of samples (ndarray of floating point values)
* sr - Sampling rate

In [ ]:
# Display basic information about the audio file
duration = librosa.get_duration(y=y, sr=sr)
print(f"Audio duration: {duration:.2f} seconds")
print(f"Sample rate: {sr} Hz")
print(f"Number of samples: {len(y)}")
print(f"Shape of audio array:{y.shape}")

These properties tell us:

Duration: The length of the audio in seconds

Sample rate: The number of samples per second (typically 22050 Hz for librosa)

Number of samples: Total data points in the audio signal

Shape: The dimensions of the numpy array (should be 1D for mono audio)

## What is Sampling rate?
Sampling rate is the number of samples we take of the audio signal per second. In simpler words, it is the number of snapshots or peeks we take at the signal. The more snapshots you take, the more accurately the discrete 0s and 1s of the digital representation correspond to the infinitely smooth and continuous original analog wave front.

In [ ]:
# t_fine represents the continuous "analog" signal
t_fine = np.linspace(0, 2 * np.pi, 500)
y_analog = np.sin(t_fine) + 0.5 * np.sin(2 * t_fine) # A complex wave

# t_sample represents our "snapshots in time" (Sampling Rate)
t_sample = np.linspace(0, 2 * np.pi, 15)
y_sample = np.sin(t_sample) + 0.5 * np.sin(2 * t_sample)

plt.figure(figsize=(8, 6))

plt.plot(t_fine, y_analog, color='lightgray', linewidth=3, label='Analog Signal')

markerline, stemlines, baseline = plt.stem(t_sample, y_sample, linefmt='r-', markerfmt='r^', basefmt='k-')
plt.setp(stemlines, 'linewidth', 2) # Make the "arrows" thicker

plt.axhline(0, color='black', linewidth=1.5) # X-axis
plt.axvline(0, color='black', linewidth=1.5) # Y-axis
plt.title('Sample Rate: "Snapshots in Time"', fontsize=14)
plt.xlabel('Time t', loc='right', fontsize=12, fontweight='bold')
plt.ylabel('Amplitude x', loc='top', fontsize=12, fontweight='bold', rotation=0)
plt.xticks([]) # Hide numbers to keep it clean like the diagram
plt.yticks([])
plt.grid(axis='x', linestyle='--', alpha=0.7) # Vertical dashed lines

plt.show()

In [ ]:
def plot_quantization(ax, sampling_rate, bit_depth, title):
    t_fine = np.linspace(0, 1, 1000)
    y_analog = np.sin(2 * np.pi * t_fine)
    
    # 1. Sampling (Time Domain)
    t_sample = np.linspace(0, 1, sampling_rate)
    y_sampled = np.sin(2 * np.pi * t_sample)
    
    # 2. Quantization (Amplitude Domain/Bit Depth)
    levels = 2**bit_depth
    y_quantized = np.round((y_sampled + 1) * (levels - 1) / 2)
    y_quantized = y_quantized * 2 / (levels - 1) - 1
    
    # Plotting
    ax.plot(t_fine, y_analog, color='lightgray', label='Analog Wave', linewidth=1)
    ax.step(t_sample, y_quantized, where='mid', color='blue', label='Digital Signal', linewidth=2)
    ax.scatter(t_sample, y_quantized, color='red', s=10) # Sampling points
    
    ax.set_title(title)
    ax.set_ylim(-1.2, 1.2)
    ax.grid(True, alpha=0.3)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

# Low Quality: Low Sample Rate (12), Low Bit Depth (3 bits = 8 levels)
plot_quantization(ax1, 12, 3, "Low Quality\n(Large Error / Blocky)")

# High Quality: High Sample Rate (50), High Bit Depth (8 bits = 256 levels)
plot_quantization(ax2, 50, 8, "High Quality\n(Less Error / Smooth)")

plt.tight_layout()
plt.show()

## Sample Rate (Along the X-axis)
Determines the **width of your digital steps**.
If the sample rate is low, the steps are wide. This means the system is ''blind'' to what happens between those snapshots. Fast, high-frequency oscillations (like a hi-hat in music or a quick spike in data) are completely missed because they occur and disappear between the samples.

## Amplitude (Along the Y-axis)
Amplitude precision (often called Bit Depth) determines the **height of your digital steps**.
It defines how many "levels" or "slots" are available to measure the vertical height of the wave.
If you have low amplitude precision, the system has to "round" the wave’s true height to the nearest available level. This rounding creates the vertical "staircase" jumps. The difference between the actual wave and the rounded digital level is called quantization error, which appears as noise or distortion in the final signal.

### Key Takeaway: Take a high enough sampling rate! :)


### Listening to the audio
We can use IPython.display.Audio to play the audio in our notebook as shown below.

In [ ]:
# Play the audio file
ipd.display(ipd.Audio(y, rate=sr))

### Listening to the effects that sampling rate has

In [ ]:
import IPython.display as ipd
import numpy as np
import scipy
from scipy.io import wavfile
from scipy import signal

def bandpass(x,lo,hi):
    X = scipy.fft.dct(x)
    N = len(X)
    X[0:int(lo*N*2)] = 0
    X[int(hi*N*2):] = 0
    return scipy.fft.idct(X)

audio_path = librosa.example('libri1')   # a clear speech recording
y1, sr1 = librosa.load(audio_path)
ipd.display(ipd.HTML('Original (0 to 22050 Hz)'))
ipd.display(ipd.Audio(y1, rate=sr1))
ipd.display(ipd.HTML('Narrowband (300 Hz to 3.3 kHz)'))
ipd.display(ipd.Audio(bandpass(y1, 300/sr1, 3300/sr1),rate=sr1))
ipd.display(ipd.HTML('Wideband (50 Hz to 7 kHz)'))
ipd.display(ipd.Audio(bandpass(y1, 50/sr1, 7000/sr1),rate=sr1))
ipd.display(ipd.HTML('Superwideband (50 Hz to 16 kHz)'))
ipd.display(ipd.Audio(bandpass(y1, 50/sr1, 16000/sr1),rate=sr1))
ipd.display(ipd.HTML('Fullband (50 Hz to 22 kHz)'))
ipd.display(ipd.Audio(bandpass(y1, 50/sr1, 22000/sr1),rate=sr1))

## Visualize the waveform
Now, we will check the basic representation of the audio signal with respect to time.

An audio signal is then represented by a sequence of numbers $x_n$ which represent the relative air pressure at time-instant $n \in N$



For that, we will use librosa.display.waveshow(sample, sampling_rate)

In [ ]:
# Visualize the waveform
plt.figure(figsize=(14, 5))
librosa.display.waveshow(y, sr=sr)
print(type(librosa.display.waveshow(y, sr=sr)))
plt.title("Audio Waveform")
plt.xlabel("Time (secs)")
plt.ylabel("Amplitude")
plt.show()

Interpretation of  the waveform:
* x-axis represents time
* y-axis represents the amplitude of the signal
* Louder parts of the audio will have larger amplitudes (taller waves)
* Quiet parts will be closer to the center line

# What is Windowing? And why?
Let's first understand the why.
Our audio signal is a continuous variation of numbers (amplitude) with respect to time. To extract information from this signal, we must therefore split the signal into sufficiently short segments. In other words, we want to extract segments which are short enough that the properties of the speech signal does not have time change within that segment.

A classical method to split the input into temporal segments is known as **Windowing**

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Load a speech clip from librosa (resampled to 8 kHz, like the original demo)
data, fs = librosa.load(librosa.example('libri1'), sr=8000)   # fs = sampling rate

window_length_ms = 30
window_length = int(np.round(fs * window_length_ms / 1000))

n = np.linspace(0.5, window_length - 0.5, num=window_length)

# windowing function
windowing_fn = np.sin(np.pi * n / window_length) ** 2   # sine (Hann-like) window

# pick a window from the middle of the signal (portable: no hard-coded index)
start = len(data) // 2
datawin = data[start:start + window_length]
datawin = datawin / np.max(np.abs(datawin))   # normalize

plt.plot(n * 1000 / fs, datawin)
plt.xlabel('Time (ms)'); plt.ylabel('Amplitude')
plt.title('A window of the signal (no windowing function applied)')
plt.axis([-10., 45., -1., 1.]); plt.tight_layout(); plt.show()

nx = np.concatenate(([-1000, 0.], n, [window_length, window_length + 1000]))
datax = np.concatenate(([0., 0.], datawin, [0., 0.]))
plt.plot(nx * 1000 / fs, datax)
plt.xlabel('Time (ms)'); plt.ylabel('Amplitude')
plt.title('A rectangular window looks as if it had a discontinuity at the borders')
plt.axis([-10., 45., -1., 1.]); plt.tight_layout(); plt.show()

# Spectrogram
## What is a Spectrogram?
A Spectrogram is a visual representation of an audio signal. This representation shows how the frequency-domain content of the signal changes over time. The graph's x-axis denotes the time and the y-axis denotes the frequency of the signal.

**Note**: Frequency domain content is also referred to as Spectral content


### What are the steps involved in Spectrogram?

1. Short Time Fourier Transform (**STFT**) - Converts the signal from time-domain to the frequency domain.
2. Windowing
3. FT each frame

## Fourier Transform

### Discrete Fourier Transform (DFT)
The DFT transforms our signal from the time-domain to the frequency domain. We use the formula:

$ X_{k}=\sum _{n=0}^{N-1}x_{n}\cdot e^{-i\cdot 2\pi \cdot k\cdot n/N}$

where:

$k$: Current frequency index $ 0\le k\le N-1 $

$N$: Total number of samples

$n$: Current sample index

$x_{n}$: Value of the signal in the time domain at index $n$

$X_{k}$: Frequency-domain coefficient (complex number containing amplitude and phase)

$i$: Imaginary unit $\sqrt{-1}$. 


Another form of the DFT (using Euler's formula)

$ X_{k}=\sum _{n=0}^{N-1}x_{n} \cdot \cos(2\pi \cdot k\cdot n/N) - i\sin(2\pi \cdot k\cdot n/N)$


The DFT has a time complexity of $O(n^2)$ which makes it extremely slow for larger number of samples (Say, 10000).  Due to this, we move on to Fast Fourier Transform (FFT).

## Fast Fourier Transform (FFT).

**FFT** relies on the fact that a DFT of size $N$ can be rewritten as the sum of two DFTs of size $N/2$.

$$X_k = E_k + e^{-\frac{i 2\pi}{N} k} \cdot O_k$$

Where:

$E_k$ is the DFT of the even-indexed elements.

$O_k$ is the DFT of the odd-indexed elements.

$e^{-\frac{i 2\pi}{N} k}$ is called the Twiddle Factor.

### Short Time Fourier Transform (STFT)



In [ ]:
# Compute the spectrogram
#What is the Fourier Transform
#Use Short-Time Fourier Transform
#Break audio to overlapping frame
#Compute fourier transform for each frame
D = librosa.stft(y) #STFT
print(f"Datatype after using STFT: {type(D)}") #Array or Matrix 2D
S_db = librosa.amplitude_to_db(np.abs(D), ref=np.max) #Amplitude to Decibel (dB)
print(f"Datatype after using Amp to dB: {type(S_db)}") #Array or Matrix 2D

#Display spectrogram
plt.figure(figsize=(14, 5))
librosa.display.specshow(S_db, sr=sr, x_axis='time', y_axis='hz')
plt.colorbar(format='%+2.0f dB')
plt.title('Spectrogram')
plt.show()

## Mel Spectrogram

Studies have shown that humans do not perceive frequencies on a linear scale. We are better at detecting differences in lower frequencies than higher frequencies. For example, we can easily tell the difference between 500 and 1000 Hz, but we will hardly be able to tell a difference between 10,000 and 10,500 Hz, even though the distance between the two pairs are the same.

In 1937, Stevens, Volkmann, and Newmann proposed a unit of pitch such that equal distances in pitch sounded equally distant to the listener. This is called the mel scale. We perform a mathematical operation on frequencies to convert them to the mel scale.

Mathematically, $$ m = 2595 \times log_{10}(1 + \frac{f}{700}) $$

In [ ]:
# Define frequencies (Hz)
hz = np.linspace(0, 8000, 1000)

# Convert to Mels
mels = librosa.hz_to_mel(hz)

# Plot
plt.plot(hz, mels)
plt.title('Mel Scale vs Frequency')
plt.xlabel('Frequency (Hz)')
plt.ylabel('Mel Scale')
plt.grid(True)
plt.show()


**Mel Spectrogram**

Similar to the spectrogram for visual representation, the Mel Spectrogram is a visual representation of the audio signal in the Mel scale.

In [ ]:
#Compute Mel Spectrogram
#What is the Mel scale?
#f = 700(10^(m/295) - 1)
S = librosa.feature.melspectrogram(y=y, sr=sr, n_mels=128)
print(f"Datatype after using converting to Melspectrogram: {type(S)}") #Array or Matrix 2D
S_db_mel = librosa.power_to_db(S, ref=np.max)
print(f"Datatype after using converting power to db: {type(S_db_mel)}") #Array or Matrix 2D

#Display the mel spectrogram
plt.figure(figsize=(14, 5))
librosa.display.specshow(S_db_mel, sr=sr, x_axis='time', y_axis='mel')
plt.colorbar(format='%+2.0f dB')
plt.title('Mel Spectrogram')
plt.show()

---
# Interlude — A Gallery of Sounds

Now that we can read a spectrogram, let's tour several *kinds* of sound and see how each one looks. Different sounds make different features stand out, which is exactly why we have so many features. Each entry below is **layered**: a plain-English *intuition* line, then a short **"The math"** block for the formal grounding.

In [ ]:
# --- synthesis helpers for the gallery ---
SR = 22050
np.random.seed(0)   # reproducible noise

def pure_tone(freq=440, dur=2.0, sr=SR):
    t = np.linspace(0, dur, int(dur * sr), endpoint=False)
    return 0.5 * np.sin(2 * np.pi * freq * t)

def two_tones(f1=440, f2=660, dur=2.0, sr=SR):
    t = np.linspace(0, dur, int(dur * sr), endpoint=False)
    return 0.4 * (np.sin(2 * np.pi * f1 * t) + np.sin(2 * np.pi * f2 * t))

def chirp(f0=200, f1=8000, dur=3.0, sr=SR):
    """A tone that smoothly sweeps from f0 up to f1."""
    t = np.linspace(0, dur, int(dur * sr), endpoint=False)
    return np.sin(2 * np.pi * (f0 * t + (f1 - f0) / (2 * dur) * t ** 2))

def percussion(dur=2.5, sr=SR):
    """A simple kick + snare pattern to demonstrate sharp transients."""
    n = int(dur * sr); y = np.zeros(n)
    def kick(t0):
        i = int(t0 * sr); L = int(0.18 * sr); tt = np.arange(L) / sr
        f = 110 * np.exp(-25 * tt)
        y[i:i+L] += np.sin(2*np.pi*np.cumsum(f)/sr) * np.exp(-18*tt)
    def snare(t0):
        i = int(t0 * sr); L = int(0.12 * sr); tt = np.arange(L) / sr
        y[i:i+L] += np.random.randn(L) * np.exp(-30*tt) * 0.7
    for b in range(4):
        kick(b*0.6); snare(b*0.6 + 0.3)
    return y / np.max(np.abs(y) + 1e-9)

### 1. Pure tone — 440 Hz (the note "A")
**Intuition.** The simplest sound: one steady frequency, a plain hum.
**The math.** A sampled sinusoid $x[n]=A\sin(2\pi f n/f_s+\varphi)$ has a spectrum that is a single line at $f$ — one flat horizontal stripe.

In [ ]:
play_and_show(pure_tone(440), SR, "Pure tone (440 Hz)", fmax=4000)

### 2. Two tones together
**Intuition.** Add a second frequency (660 Hz) and the sound gets richer, chord-like.
**The math.** The Fourier transform is **linear**, so the spectrum of a sum is the sum of the spectra — *two* lines. Any sound is a stack of simple tones like these.

In [ ]:
play_and_show(two_tones(440, 660), SR, "Two tones (440 + 660 Hz)", fmax=4000)

### 3. Chirp — a frequency sweep
**Intuition.** The pitch slides up from 200 Hz to 8 kHz — a rising "wheeee", a clean diagonal line.
**The math.** Instantaneous frequency is the derivative of phase. With $\phi(t)=2\pi(f_0 t+\tfrac{f_1-f_0}{2T}t^2)$, $\ f(t)=\tfrac{1}{2\pi}\tfrac{d\phi}{dt}=f_0+\tfrac{f_1-f_0}{T}t$ — a straight line. This is the ideal signal for **aliasing**: by the **Nyquist limit**, anything above $f_s/2$ folds back to a lower frequency.

In [ ]:
play_and_show(chirp(200, 8000, 3.0), SR, "Linear chirp (200 Hz → 8 kHz)")

### 4. Percussion — kick & snare
**Intuition.** Thump, *tss*, thump, *tss* — kicks are low blobs, snares are tall broadband streaks.
**The math.** Percussion is dominated by **transients**. The time–frequency **uncertainty principle** ($\Delta t\,\Delta f \gtrsim \tfrac{1}{4\pi}$) says a sound sharp in *time* must be spread in *frequency* — which is why a snare smears across the band, and why ZCR and RMS react so strongly to it.

In [ ]:
play_and_show(percussion(), SR, "Percussion (kick + snare)", fmax=6000)

### 5. A real instrument — trumpet
**Intuition.** A real recording shows a *ladder* of evenly spaced lines plus a little breath noise.
**The math.** A sustained pitched instrument is **harmonic**: energy at integer multiples $f_k=k f_0$. The relative heights of those harmonics are what we hear as **timbre**.

In [ ]:
y_tr, sr_tr = librosa.load(librosa.example("trumpet"))
play_and_show(y_tr, sr_tr, "Trumpet (real recording)", fmax=8000)

### 6. Nature — birdsong
**Intuition.** A solo robin: rapid swooping curves high up, unlike steady instrument stripes.
**The math.** Birdsong is **non-stationary** and strongly **frequency-modulated** — only a time–frequency view (the spectrogram) captures it; a single transform of the whole clip would blur it away.

In [ ]:
y_bird, sr_bird = librosa.load(librosa.example("robin"))
play_and_show(y_bird, sr_bird, "Robin birdsong (real recording)", fmax=10000)

### 7. Singing voice
**Intuition.** A sustained "ahh" with a gentle wobble; a tall stack of harmonics, each *wiggling*.
**The math.** The wiggle is **vibrato** — a slow frequency modulation $F_0(t)=F_0(1+d\sin 2\pi f_{\text{vib}}t)$. Each harmonic at $kF_0$ wiggles $k$ times as much, so the wobble grows toward the top. Singing is a sustained, musical version of the voiced speech we look at next.

In [ ]:
def singing(dur=2.5, sr=SR, base=220):
    n = int(dur * sr); t = np.arange(n) / sr
    f0 = base * (1 + 0.03 * np.sin(2 * np.pi * 5.5 * t))   # 5.5 Hz vibrato
    phase = np.cumsum(f0) / sr
    y = sum(np.sin(2 * np.pi * k * phase) / k for k in range(1, 30))  # harmonics
    for f in [700, 1220, 2600]:                            # vowel "ah" formants
        r = np.exp(-np.pi * 90 / sr); th = 2 * np.pi * f / sr
        y = lfilter([1 - r], [1, -2*r*np.cos(th), r*r], y)
    env = np.minimum(1, np.minimum(t / 0.1, (dur - t) / 0.2))
    return (y * env) / np.max(np.abs(y) + 1e-9)

play_and_show(singing(), SR, "Singing voice (vowel with vibrato)", fmax=4000)

# Time-domain Features

Time-domain features are extracted directly from the audio waveform. These features capture temporal characteristics of the audio signal and are often computationally efficient to calculate. In this section, we’ll explore two important time-domain features: Root Mean Square (RMS) Energy and Zero Crossing Rate (ZCR).

## RMS Energy - The square root of the mean of the square.

RMS is (to engineers anyway) a meaningful way of calculating the average of values over a period of time. With audio, the signal value (amplitude) is squared, averaged over a period of time, then the square root of the result is calculated. The result is a value, that when squared, is related (proportional) to the effective power of the signal.

In [ ]:
#What is frame and hop length
#Frame Length = Size the frame while scanning the signal
#Hop Length = Length difference between two frames
# Calculate RMS energy
frame_length = 2048
hop_length = 512
rms = librosa.feature.rms(y=y, frame_length=frame_length, hop_length=hop_length)
print(f"Datatype of rms: {type(rms)}")

# Plot RMS energy
plt.figure(figsize=(14, 5))
times = librosa.times_like(rms, sr=sr, hop_length=hop_length)
plt.plot(times, rms[0])
plt.title("Root Mean Square Energy")
plt.xlabel("Time (seconds)")
plt.ylabel("RMS Energy")
plt.show()

## Zero Crossing Rate

The Zero-Crossing Rate (ZCR) of an audio frame is the rate of sign-changes of the signal during the frame. In other words, it is the number of times the signal changes value, from positive to negative and vice versa, divided by the length of the frame.

In more simpler words, it is the rate at which the signal crosses zero.

In [ ]:
# Calculate Zero Crossing Rate
zcr = librosa.feature.zero_crossing_rate(y, frame_length=frame_length, hop_length=hop_length)
print(f"Datatype of zcr: {type(zcr)}")

# Plot Zero Crossing Rate
plt.figure(figsize=(14, 5))
times = librosa.times_like(zcr, sr=sr, hop_length=hop_length)
plt.plot(times, zcr[0])
plt.title("Zero Crossing Rate")
plt.xlabel("Time (seconds)")
plt.ylabel("Zero Crossing Rate")
plt.show()

### RMS + ZCR Combined

In [ ]:
plt.figure(figsize=(14, 8))

plt.subplot(2, 1, 1)
librosa.display.waveshow(y, sr=sr, alpha=0.5)
plt.plot(times, rms[0] / rms.max(), color='r', label='RMS Energy')
plt.title("Waveform and RMS Energy")
plt.legend()

plt.subplot(2, 1, 2)
librosa.display.waveshow(y, sr=sr, alpha=0.5)
plt.plot(times, zcr[0], color='g', label='Zero Crossing Rate')
plt.title("Waveform and Zero Crossing Rate")
plt.legend()

plt.tight_layout()
plt.show()

#### Feature Statistics

We often need to summarize these data for use in Machine Learning models.

In [ ]:
# Calculate statistics
rms_mean = np.mean(rms)
rms_std = np.std(rms)
zcr_mean = np.mean(zcr)
zcr_std = np.std(zcr)

print(f"RMS Energy - Mean: {rms_mean:.4f}, Std Dev: {rms_std:.4f}")
print(f"Zero Crossing Rate - Mean: {zcr_mean:.4f}, Std Dev: {zcr_std:.4f}")

---
# Speech Up Close — Voiced vs. Unvoiced, Male vs. Female

We just used RMS energy and the zero-crossing rate. Here is where they earn their keep: telling apart the two basic classes of speech sound.

**Intuition.** Speech is a *buzz* or a *hiss* (the source) shaped by the tube of your mouth and throat (the filter).

**The math — the source-filter model.** Over a short frame, speech is the source $e[n]$ convolved with the vocal-tract response $v[n]$: $\ s[n]=(e*v)[n]\Leftrightarrow S=E\cdot V$. Only the **source** changes between the two classes:
- **Voiced** (vowels, "m", "z"): the folds buzz periodically → source is an impulse train → spectrum is a **comb of harmonics** at multiples of $F_0$, with $|V|$ shaping the envelope.
- **Unvoiced** (hisses "s", "f", "sh"): the source is **white noise** (flat spectrum) → output is just $|V|^2$ — broadband, no harmonic lines.

**Formants** are the resonances of $V$, built as 2-pole resonators — exactly what `vocal_tract()` implements: $\ H(z)=\frac{1-r}{1-2r\cos\theta\,z^{-1}+r^2 z^{-2}}$ with $\theta=2\pi f_{\text{formant}}/f_s$.

In [ ]:
def vocal_tract(x, formants, sr=SR, bw=90):
    """Colour a source signal with vocal-tract resonances (formants)."""
    for f in formants:
        r = np.exp(-np.pi * bw / sr); th = 2 * np.pi * f / sr
        x = lfilter([1 - r], [1, -2*r*np.cos(th), r*r], x)
    return x / (np.max(np.abs(x)) + 1e-9)

def voiced(f0, formants=(700, 1220, 2600), dur=1.2, sr=SR):
    n = int(dur * sr)
    src = np.zeros(n); src[::int(sr / f0)] = 1.0     # buzzing vocal folds
    return vocal_tract(src, formants, sr)

def unvoiced(formants=(1400, 4500), dur=1.2, sr=SR):
    src = np.random.randn(int(dur * sr)) * 0.3       # turbulent air = noise
    return vocal_tract(src, formants, sr)

### Voiced vs. unvoiced — see it and hear it

In [ ]:
v = voiced(150)        # a voiced vowel "ahh"
u = unvoiced()         # an unvoiced hiss "sss"

print("Voiced vowel (ahh):"); ipd.display(ipd.Audio(v, rate=SR))
print("Unvoiced hiss (sss):"); ipd.display(ipd.Audio(u, rate=SR))

fig, ax = plt.subplots(1, 2, figsize=(13, 4))
for a, sig, title, fmax in [(ax[0], v, "Voiced 'ahh' — harmonic bands", 4000),
                            (ax[1], u, "Unvoiced 'sss' — broadband noise", 8000)]:
    D = librosa.amplitude_to_db(np.abs(librosa.stft(sig)), ref=np.max)
    librosa.display.specshow(D, sr=SR, x_axis="time", y_axis="hz", ax=a, cmap="magma")
    a.set_ylim(0, fmax); a.set_title(title)
plt.tight_layout(); plt.show()

**Intuition.** Voiced = neat horizontal stripes (it has pitch); unvoiced = a fuzzy wash (no pitch). Now confirm with *numbers*.
**The math — zero-crossing rate.** $\ \text{ZCR}=\frac{1}{2N}\sum_{n=1}^{N-1}|\operatorname{sgn}(x[n])-\operatorname{sgn}(x[n-1])|$. For energy near a frequency $f$, $\ \text{ZCR}\approx 2f/f_s$. Unvoiced energy sits high → many crossings → high ZCR. **RMS** $=\sqrt{\tfrac1N\sum x[n]^2}$ measures loudness; voiced usually carries more.

In [ ]:
def zcr_rms(sig, sr=SR):
    zcr = librosa.feature.zero_crossing_rate(sig)[0]
    rms = librosa.feature.rms(y=sig)[0]
    return zcr, rms

zv, rv = zcr_rms(v); zu, ru = zcr_rms(u)
print(f"Voiced   -> mean ZCR = {zv.mean():.3f},  mean RMS = {rv.mean():.3f}")
print(f"Unvoiced -> mean ZCR = {zu.mean():.3f},  mean RMS = {ru.mean():.3f}")

plt.figure(figsize=(11, 4))
plt.plot(librosa.times_like(zv), zv, label="ZCR — voiced")
plt.plot(librosa.times_like(zu), zu, label="ZCR — unvoiced")
plt.xlabel("Time (s)"); plt.ylabel("Zero-Crossing Rate")
plt.title("Unvoiced sounds cross zero far more often"); plt.legend(); plt.grid(alpha=0.3)
plt.show()

The higher ZCR (and lower energy) of the hiss is enough to build a basic **voiced/unvoiced detector** — the kind used inside speech recognisers.

### Male vs. female voice
**Intuition.** Same vowel, different **pitch**: ~110–150 Hz (male) vs ~190–250 Hz (female). The harmonic stripes spread apart as pitch rises.
**The math.** Harmonic spacing equals $F_0$, so higher pitch widens the gaps. There is also a physical effect: a vocal tract of length $L$ (closed–open tube) resonates at $f_k=(2k-1)c/4L$, so a shorter (female-average) tract pushes the **formants** up ~15–20% too — both pitch *and* timbre shift.

In [ ]:
male   = voiced(120)   # lower pitch
female = voiced(220)   # higher pitch

print("Male-range voice (F0 ≈ 120 Hz):");   ipd.display(ipd.Audio(male, rate=SR))
print("Female-range voice (F0 ≈ 220 Hz):"); ipd.display(ipd.Audio(female, rate=SR))

fig, ax = plt.subplots(1, 2, figsize=(13, 4))
for a, sig, title in [(ax[0], male, "Male voice (F0 ≈ 120 Hz) — closely spaced harmonics"),
                      (ax[1], female, "Female voice (F0 ≈ 220 Hz) — widely spaced harmonics")]:
    D = librosa.amplitude_to_db(np.abs(librosa.stft(sig)), ref=np.max)
    librosa.display.specshow(D, sr=SR, x_axis="time", y_axis="hz", ax=a, cmap="magma")
    a.set_ylim(0, 3500); a.set_title(title)
plt.tight_layout(); plt.show()

We can *measure* the pitch with `librosa.pyin` (a probabilistic version of the YIN autocorrelation method) and confirm the two differ.

In [ ]:
def estimate_f0(sig, sr=SR):
    f0, voiced_flag, _ = librosa.pyin(sig, fmin=80, fmax=400, sr=sr)
    return np.nanmean(f0)

print(f"Estimated F0 (male-range):   {estimate_f0(male):.1f} Hz")
print(f"Estimated F0 (female-range): {estimate_f0(female):.1f} Hz")

### Try it yourself — record your own voice (Colab)

Run the next cell, then call `record_audio()`. Your browser will ask for **microphone permission** — allow it and speak for a few seconds. Try a long **"aaah"** (voiced) and a long **"sssss"** (unvoiced), and compare the spectrograms and the ZCR/F0 numbers above.

In [ ]:
# --- Colab microphone recording ---
from IPython.display import Javascript
from google.colab import output
from base64 import b64decode

_RECORD_JS = """
const sleep = ms => new Promise(r => setTimeout(r, ms));
async function record(time) {
  const stream = await navigator.mediaDevices.getUserMedia({audio: true});
  const rec = new MediaRecorder(stream);
  const chunks = [];
  rec.ondataavailable = e => chunks.push(e.data);
  rec.start();
  await sleep(time);
  rec.stop();
  await new Promise(r => rec.onstop = r);
  stream.getTracks().forEach(t => t.stop());
  const blob = new Blob(chunks);
  const reader = new FileReader();
  reader.readAsDataURL(blob);
  await new Promise(r => reader.onloadend = r);
  return reader.result;
}
"""

def record_audio(seconds=4):
    """Record `seconds` of audio from the mic and return (y, sr)."""
    display(Javascript(_RECORD_JS))
    data_uri = output.eval_js(f"record({int(seconds * 1000)})")
    binary = b64decode(data_uri.split(",")[1])
    with open("recording.webm", "wb") as f:
        f.write(binary)
    y, sr = librosa.load("recording.webm", sr=SR)   # ffmpeg (in Colab) decodes webm
    return y, sr

print("Defined record_audio(). Run the next cell to record.")

In [ ]:
# Record ~4 seconds, then play it back, show its spectrogram, and estimate your pitch.
y_me, sr_me = record_audio(seconds=4)
play_and_show(y_me, sr_me, "Your recording")

z, r = zcr_rms(y_me, sr_me)
print(f"Your mean ZCR: {z.mean():.3f}   |   mean RMS: {r.mean():.3f}")
try:
    print(f"Your estimated pitch (F0): {estimate_f0(y_me, sr_me):.1f} Hz")
except Exception as e:
    print("Pitch estimate skipped:", e)

# Frequency domain features

While time-domain features talks about the temporal characteristics of audio/speech signals, frequency-domain features provide information about the **spectral content** (energy, frequency) of the signal.
In this section, we’ll explore three important frequency-domain features:

1. Short-time Fourier Transform (STFT)
2. Spectral Centroid
3. Spectral Rolloff

## Short Time Fourier Transform (STFT)

The STFT uses the DFT alongside a windowing function $w_n$. Mathematically,

$STFT(X_{k}, w_{n}) =\sum _{n=0}^{N-1}x_{n} \cdot w_n \cdot e^{-i\cdot 2\pi \cdot k\cdot n/N}$

A further parallel with a spectrum is that the output of the STFT is complex-valued, though where the spectrum is a vector, the STFT output is a matrix. As a consequence, we cannot directly visualize the complex-valued output. Instead, STFTs are usually visualized using their log-spectra, $20 \cdot log(X)$. Such 2 dimensional log-spectra can then be visualized with a heat-map known as a spectrogram.

In [ ]:
# Compute STFT
n_fft = 2048
hop_length = 512
D = librosa.stft(y, n_fft=n_fft, hop_length=hop_length)

# Convert to dB scale
D_db = librosa.amplitude_to_db(np.abs(D), ref=np.max)

# Visualize the spectrogram
plt.figure(figsize=(14, 6))
librosa.display.specshow(D_db, sr=sr, x_axis='time', y_axis='hz', hop_length=hop_length)
plt.colorbar(format='%+2.0f dB')
plt.title('Short-time Fourier Transform (STFT)')
plt.show()

#### Band Energy Ratio

Compare energy in frequency bands. Measure how dominant lower frequencies are.

In [ ]:
n_fft = 2048
hop_length = 512

S = np.abs(librosa.stft(y, n_fft=n_fft, hop_length=hop_length))**2
freqs = librosa.fft_frequencies(sr=sr, n_fft=n_fft)

low_band = (0, 500)        # 0–500 Hz
high_band = (500, 4000)    # 500–4000 Hz

def band_energy_ratio(S, freqs, band):
    band_idx = np.logical_and(freqs >= band[0], freqs <= band[1])
    band_energy = np.sum(S[band_idx, :], axis=0)
    total_energy = np.sum(S, axis=0)
    return band_energy / (total_energy + 1e-10)

ber_low = band_energy_ratio(S, freqs, low_band)
ber_high = band_energy_ratio(S, freqs, high_band)

times = librosa.frames_to_time(
    np.arange(S.shape[1]),
    sr=sr,
    hop_length=hop_length
)

plt.figure(figsize=(12, 5))
plt.plot(times, ber_low, label="Low Band (0–500 Hz)")
plt.plot(times, ber_high, label="High Band (500–4000 Hz)")
plt.xlabel("Time (s)")
plt.ylabel("Band Energy Ratio")
plt.title("Band Energy Ratio over Time")
plt.legend()
plt.grid(True)
plt.show()


## Spectral Centroid

Measure of centre of mass of sounds in frequency terms. Think of it as the "brightness of sound". In other words, it shows us which frequency is dominating at which point of time.

Spectral Centroid $= \frac{\sum f \cdot m(f)}{\sum m(f)}$

where, 

$f$ - Frequency

$m(f)$ - Magnitude of $f$

In [ ]:
# Compute spectral centroid
centroid = librosa.feature.spectral_centroid(y=y, sr=sr, hop_length=hop_length)
print(f"Datatype of Spectral Centroid: {type(centroid)}")

# Visualize spectral centroid
plt.figure(figsize=(14, 6))
times = librosa.times_like(centroid, sr=sr, hop_length=hop_length)
plt.semilogy(times, centroid[0], label='Spectral Centroid')
plt.ylabel('Hz')
plt.xlabel('Time')
plt.legend()
plt.title('Spectral Centroid')
plt.show()

## Spectral Rolloff

Frequency below which a certain percentage of total spectral energy lies. Another measure of the spectral shape of the sound.

In [ ]:
# Compute spectral rolloff
#Default rolloff used 85%
rolloff = librosa.feature.spectral_rolloff(y=y, sr=sr, hop_length=hop_length)
print(f"Datatype of rolloff: {type(rolloff)}")

# Visualize spectral rolloff
plt.figure(figsize=(14, 6))
times = librosa.times_like(rolloff, sr=sr, hop_length=hop_length)
#85% of the signal’s energy lies below this frequency
plt.semilogy(times, rolloff[0], label='Spectral Rolloff')
plt.ylabel('Hz')
plt.xlabel('Time')
plt.legend()
plt.title('Spectral Rolloff')
plt.show()

### Spectral Centroid + Rolloff

In [ ]:
plt.figure(figsize=(14, 8))

# Plot spectrogram
librosa.display.specshow(D_db, sr=sr, x_axis='time', y_axis='log', hop_length=hop_length)
plt.colorbar(format='%+2.0f dB')
plt.title('STFT Spectrogram with Spectral Centroid and Rolloff')

# Plot spectral centroid on top of the spectrogram
plt.semilogy(times, centroid[0], label='Spectral Centroid', color='w')

# Plot spectral rolloff on top of the spectrogram
plt.semilogy(times, rolloff[0], label='Spectral Rolloff', color='r')

plt.ylabel('Frequency (Hz)')
plt.xlabel('Time (s)')
plt.legend(loc='upper right')
plt.grid(True, linestyle='--', alpha=0.6)
plt.tight_layout()
plt.show()

In [ ]:
# Calculate statistics
centroid_mean = np.mean(centroid)
centroid_std = np.std(centroid)
rolloff_mean = np.mean(rolloff)
rolloff_std = np.std(rolloff)

print(f"Spectral Centroid - Mean: {centroid_mean:.2f} Hz, Std Dev: {centroid_std:.2f} Hz")
print(f"Spectral Rolloff - Mean: {rolloff_mean:.2f} Hz, Std Dev: {rolloff_std:.2f} Hz")

### Mel-frequency Cepstral Coefficients (MFCCs)

Relates how humans perceive pitch and frequency content. Think of audio fingerprint and MFCCs capture that.

In [ ]:
# Compute MFCCs
n_mfcc = 13
mfccs = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=n_mfcc)
print(f"Datatype of MFCCs: {type(mfccs)}")

# Visualize MFCCs
plt.figure(figsize=(14, 6))
librosa.display.specshow(mfccs, sr=sr, x_axis='time')
plt.colorbar(format='%+2.0f dB')
plt.title('Mel-frequency Cepstral Coefficients (MFCCs)')
plt.show()

In [ ]:
plt.figure(figsize=(14, 8))
for i in range(4):  # Plot first 4 MFCCs
    plt.subplot(4, 1, i+1)
    plt.plot(librosa.times_like(mfccs), mfccs[i])
    plt.title(f'MFCC {i+1}')
    plt.xlabel('Time')
    plt.ylabel('Magnitude')
plt.tight_layout()
plt.show()

Delta MFCCs
* What does the sound look like right now?
* Then, Delta says How is the sound changing right now?

In [ ]:
# Compute delta and delta-delta MFCCs
mfccs_delta = librosa.feature.delta(mfccs)
mfccs_delta2 = librosa.feature.delta(mfccs, order=2)

# Visualize delta MFCCs
plt.figure(figsize=(14, 6))
librosa.display.specshow(mfccs_delta, sr=sr, x_axis='time')
plt.colorbar(format='%+2.0f')
plt.title('Delta MFCCs')
plt.show()

# Visualize delta-delta MFCCs
plt.figure(figsize=(14, 6))
librosa.display.specshow(mfccs_delta2, sr=sr, x_axis='time')
plt.colorbar(format='%+2.0f')
plt.title('Delta-Delta MFCCs')
plt.show()

In [ ]:
# Calculate statistics
mfcc_means = np.mean(mfccs, axis=1)
mfcc_vars = np.var(mfccs, axis=1)
delta_means = np.mean(mfccs_delta, axis=1)
delta_vars = np.var(mfccs_delta, axis=1)

# Display statistics
plt.figure(figsize=(14, 6))
plt.subplot(2, 1, 1)
plt.bar(range(1, n_mfcc + 1), mfcc_means)
plt.title('Mean of MFCCs')
plt.xlabel('MFCC Coefficient')
plt.ylabel('Mean Value')

plt.subplot(2, 1, 2)
plt.bar(range(1, n_mfcc + 1), mfcc_vars)
plt.title('Variance of MFCCs')
plt.xlabel('MFCC Coefficient')
plt.ylabel('Variance')

plt.tight_layout()
plt.show()

# Chroma Features

Chroma Features describe how much energy the audio has in each of the 12 musical pitch classes, ignoring octave.
Big idea (plain English)

Western music can be reduced to 12 notes:

C, C♯/D♭, D, D♯/E♭, E, F, F♯/G♭, G, G♯/A♭, A, A♯/B♭, B

Chroma features answer:

“At this moment, how strong is each of these 12 notes?”

So instead of tracking where a note sits in frequency (octave), chroma tracks which note it is.

In [ ]:
# Compute chroma features
chroma = librosa.feature.chroma_stft(y=y, sr=sr)

# Visualize chroma features
plt.figure(figsize=(14, 6))
librosa.display.specshow(chroma, sr=sr, x_axis='time', y_axis='chroma')
plt.colorbar()
plt.title('Chromagram')
plt.show()

In [ ]:
# Define pitch classes
pitch_classes = ['C', 'C#', 'D', 'D#', 'E', 'F', 'F#', 'G', 'G#', 'A', 'A#', 'B']

plt.figure(figsize=(14, 8))
for i in range(12):
    plt.subplot(4, 3, i+1)
    plt.plot(librosa.times_like(chroma), chroma[i])
    plt.title(pitch_classes[i])
    plt.xlabel('Time')
    plt.ylabel('Magnitude')
plt.tight_layout()
plt.show()

In [ ]:
# Compute CENS features
cens = librosa.feature.chroma_cens(y=y, sr=sr)

# Visualize CENS features
plt.figure(figsize=(14, 6))
librosa.display.specshow(cens, sr=sr, x_axis='time', y_axis='chroma')
plt.colorbar()
plt.title('Chroma Energy Normalized Statistics (CENS)')
plt.show()

In [ ]:
hop_lengths = [512, 2048, 4096]
plt.figure(figsize=(14, 12))

for i, hop_length in enumerate(hop_lengths):
    chroma = librosa.feature.chroma_stft(y=y, sr=sr, hop_length=hop_length)
    plt.subplot(3, 1, i+1)
    librosa.display.specshow(chroma, sr=sr, x_axis='time', y_axis='chroma', hop_length=hop_length)
    plt.colorbar()
    plt.title(f'Chromagram (hop_length = {hop_length})')

plt.tight_layout()
plt.show()

In [ ]:
# Calculate statistics
chroma_means = np.mean(chroma, axis=1)
chroma_vars = np.var(chroma, axis=1)

# Display statistics
plt.figure(figsize=(14, 6))
plt.subplot(2, 1, 1)
plt.bar(pitch_classes, chroma_means)
plt.title('Mean of Chroma Features')
plt.xlabel('Pitch Class')
plt.ylabel('Mean Value')

plt.subplot(2, 1, 2)
plt.bar(pitch_classes, chroma_vars)
plt.title('Variance of Chroma Features')
plt.xlabel('Pitch Class')
plt.ylabel('Variance')

plt.tight_layout()
plt.show()

In [ ]:
def extract_features(y, sr):
    """Extract a compact set of audio features and return them as a dict."""
    features = {}

    # Time-domain features
    features['rms'] = librosa.feature.rms(y=y).mean()
    features['zcr'] = librosa.feature.zero_crossing_rate(y).mean()

    # Frequency-domain features
    features['spectral_centroid'] = librosa.feature.spectral_centroid(y=y, sr=sr).mean()
    features['spectral_rolloff']  = librosa.feature.spectral_rolloff(y=y, sr=sr).mean()

    # MFCCs
    mfccs = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=13)
    for i, m in enumerate(mfccs):
        features[f'mfcc_{i+1}'] = m.mean()

    # Chroma features
    chroma = librosa.feature.chroma_stft(y=y, sr=sr)
    for i, pc in enumerate(pitch_classes):
        features[f'chroma_{pc}'] = chroma[i].mean()

    return features

# Extract features from the audio file and show them as a one-row table
audio_features = extract_features(y, sr)
df = pd.DataFrame([audio_features])
df